## Setup

In [1]:
import requests
from io import BytesIO
from PIL import Image
from statistics import median
from dataclasses import dataclass
from IPython.display import HTML
import base64

In [11]:
import pandas as pd
import re

In [3]:
## to store results
@dataclass
class ColorMeasurement:
  r: int
  g: int
  b: int
  hex_color: str

## Load Dataset

In [13]:
# Load the CSV file into a pandas DataFrame
df_blush = pd.read_csv('/content/cleaned_single_blush.csv')

# Display the first few rows to verify loading
df_blush.head()

,product_id,sku_id,variant_name,variant_description,variant_url,swatch_image_url,variant_image_url,ingredient_set_id,brand,product_name,size_value,size_unit,effective_price,normalized_size_g_ml,price_per_g_ml
0,pimprod2055883,2647881,Happy,cool pink (dewy),https://www.ulta.com/p/soft-pinch-liquid-blush...,https://media.ultainc.com/i/ulta/2647881_sw?im...,https://media.ultainc.com/i/ulta/2647881?w=500...,dcf984edd911eb352ae36c53c9d9336a74ecf620c5ee8a...,Rare Beauty,Soft Pinch Liquid Blush,0.25,oz,25.0,7.09,3.53
1,pimprod2055883,2647882,Hope,nude mauve (dewy),https://www.ulta.com/p/soft-pinch-liquid-blush...,https://media.ultainc.com/i/ulta/2647882_sw?im...,https://media.ultainc.com/i/ulta/2647882?w=500...,81cca0463ccff2929af1d5eedc7bcf8bd7546a8fd52acd...,Rare Beauty,Soft Pinch Liquid Blush,0.25,oz,25.0,7.09,3.53
2,pimprod2055883,2647880,Grateful,true red (dewy),https://www.ulta.com/p/soft-pinch-liquid-blush...,https://media.ultainc.com/i/ulta/2647880_sw?im...,https://media.ultainc.com/i/ulta/2647880?w=500...,dcf984edd911eb352ae36c53c9d9336a74ecf620c5ee8a...,Rare Beauty,Soft Pinch Liquid Blush,0.25,oz,25.0,7.09,3.53
3,pimprod2055883,2647883,Joy,muted peach (dewy),https://www.ulta.com/p/soft-pinch-liquid-blush...,https://media.ultainc.com/i/ulta/2647883_sw?im...,https://media.ultainc.com/i/ulta/2647883?w=500...,dcf984edd911eb352ae36c53c9d9336a74ecf620c5ee8a...,Rare Beauty,Soft Pinch Liquid Blush,0.25,oz,25.0,7.09,3.53
4,pimprod2055883,2647884,Lucky,hot pink (dewy),https://www.ulta.com/p/soft-pinch-liquid-blush...,https://media.ultainc.com/i/ulta/2647884_sw?im...,https://media.ultainc.com/i/ulta/2647884?w=500...,dcf984edd911eb352ae36c53c9d9336a74ecf620c5ee8a...,Rare Beauty,Soft Pinch Liquid Blush,0.25,oz,25.0,7.09,3.53


In [3]:
color_blush = pd.read_csv('/content/swatch_colors.csv')
color_blush.head()

,product_id,sku_id,swatch_image_url,image_sha256,image_width,image_height,sample_pixel_count,rgb_r,rgb_g,rgb_b,hex_color,lab_l,lab_a,lab_b,rgb_spread,extraction_method
0,pimprod2055883,2647881,https://media.ultainc.com/i/ulta/2647881_sw?im...,7c654ac1adbc7704cc7a79e0b0564ca9778c6b773708d4...,80,80,256,242,124,132,#f27c84,65.72,45.91,16.74,0.0,center_median_srgb_v1
1,pimprod2055883,2647882,https://media.ultainc.com/i/ulta/2647882_sw?im...,eced9806cc6b1c7ddb97736cd54344d6cf53f7c3667ef7...,80,80,256,232,136,145,#e88891,67.14,37.59,11.20,0.0,center_median_srgb_v1
2,pimprod2055883,2647880,https://media.ultainc.com/i/ulta/2647880_sw?im...,7946af3c8d9384922278ead7a7b928acd9baa65e712e14...,80,80,256,243,65,61,#f3413d,55.26,66.62,43.51,0.0,center_median_srgb_v1
3,pimprod2055883,2647883,https://media.ultainc.com/i/ulta/2647883_sw?im...,909cdb88040ddd17e2f26012fe28a59dce37b6f273ae51...,80,80,256,254,111,87,#fe6f57,64.27,52.96,40.25,0.0,center_median_srgb_v1
4,pimprod2055883,2647884,https://media.ultainc.com/i/ulta/2647884_sw?im...,f2af960b2c053ea45dd789ef33870a4d03f4fa75ddf955...,80,80,256,252,74,132,#fc4a84,59.22,70.50,7.83,0.0,center_median_srgb_v1


In [5]:
foundations = pd.read_csv(r"..\data\raw\foundation\20260803T213850Z\variants.csv", encoding="utf-8-sig")
foundations.head()

,product_id,sku_id,variant_type,variant_name,variant_description,variant_url,list_price,sale_price,size_text,availability,swatch_image_url,variant_image_url,ingredient_set_id
0,pimprod2057355,2651970,color,0N1 Alabaster,lightest with neutral undertones,https://www.ulta.com/p/double-wear-stay-in-pla...,52.0,NaN,1.0 oz,InStock,https://media.ultainc.com/i/ulta/2651970_sw?im...,https://media.ultainc.com/i/ulta/2651970?w=500...,7504fc4671712f74cee9be963f0736dc2a3054032beca7...
1,pimprod2057355,2651971,color,1C0 Shell,very light with cool pink undertones,https://www.ulta.com/p/double-wear-stay-in-pla...,52.0,NaN,1.0 oz,InStock,https://media.ultainc.com/i/ulta/2651971_sw?im...,https://media.ultainc.com/i/ulta/2651971?w=500...,7504fc4671712f74cee9be963f0736dc2a3054032beca7...
2,pimprod2057355,2651972,color,1C1 Cool Bone,light with cool rosy-peach undertones,https://www.ulta.com/p/double-wear-stay-in-pla...,52.0,NaN,1.0 oz,InStock,https://media.ultainc.com/i/ulta/2651972_sw?im...,https://media.ultainc.com/i/ulta/2651972?w=500...,7504fc4671712f74cee9be963f0736dc2a3054032beca7...
3,pimprod2057355,2651973,color,1N0 Porcelain,very light with neutral undertones,https://www.ulta.com/p/double-wear-stay-in-pla...,52.0,NaN,1.0 oz,InStock,https://media.ultainc.com/i/ulta/2651973_sw?im...,https://media.ultainc.com/i/ulta/2651973?w=500...,7504fc4671712f74cee9be963f0736dc2a3054032beca7...
4,pimprod2057355,2651974,color,1N1 Ivory Nude,light with neutral peach undertones,https://www.ulta.com/p/double-wear-stay-in-pla...,52.0,NaN,1.0 oz,InStock,https://media.ultainc.com/i/ulta/2651974_sw?im...,https://media.ultainc.com/i/ulta/2651974?w=500...,7504fc4671712f74cee9be963f0736dc2a3054032beca7...


In [8]:
foundation_products = pd.read_csv(r"..\data\raw\foundation\20260803T213850Z\products.csv", encoding='utf-8-sig')
foundation_products.head()

,product_id,brand,product_name,product_url,rating,review_count
0,pimprod2057355,Estée Lauder,Double Wear Stay-in-Place Longwear Matte Found...,https://www.ulta.com/p/double-wear-stay-in-pla...,4.3,9902
1,xlsImpprod5770257,IT Cosmetics,CC+ Cream with SPF 50+,https://www.ulta.com/p/cc-cream-with-spf-50-xl...,4.3,22003
2,pimprod2051406,MAC,Studio Fix Powder Plus Foundation,https://www.ulta.com/p/studio-fix-powder-plus-...,3.8,3448
3,pimprod2046452,KYLIE COSMETICS,Skin Tint Blurring Elixir Foundation,https://www.ulta.com/p/skin-tint-blurring-elix...,4.5,1485
4,pimprod2044825,MAC,Studio Fix Fluid SPF15 24HR Matte Foundation +...,https://www.ulta.com/p/studio-fix-fluid-spf15-...,4.2,2325


## Cleaning Foundation Dataset

In [ ]:
foundations['price'] = foundations['sale_price'].fillna(foundations['list_price']) # created a new price column

## Merge foundation products to foundation variants
foundations = foundations.merge(
    foundation_products[['brand', 'product_name', 'product_id']]
    , on = 'product_id'
    , how = 'left'
    , validate = 'many_to_one'
)

foundations.loc[foundations['size_text'].str.contains('Color:'), 'size_text'] = "1.0 oz" # replacing with "0.1 oz" if contains "Color:"


In [17]:
# Regex captures floating/integer numbers and the trailing unit string
pattern = r'^\s*(\d*\.?\d+)\s*([a-zA-Z]+|\S+)?'

extracted = foundations['size_text'].str.extract(pattern)
foundations['size_value'] = pd.to_numeric(extracted[0], errors='coerce')
foundations['size_unit'] = extracted[1].str.lower().str.strip()

foundations.head()

,product_id,sku_id,variant_type,variant_name,variant_description,variant_url,size_text,availability,swatch_image_url,variant_image_url,ingredient_set_id,price,brand,product_name,size_value,size_unit
0,pimprod2057355,2651970,color,0N1 Alabaster,lightest with neutral undertones,https://www.ulta.com/p/double-wear-stay-in-pla...,1.0 oz,InStock,https://media.ultainc.com/i/ulta/2651970_sw?im...,https://media.ultainc.com/i/ulta/2651970?w=500...,7504fc4671712f74cee9be963f0736dc2a3054032beca7...,52.0,Estée Lauder,Double Wear Stay-in-Place Longwear Matte Found...,1.0,oz
1,pimprod2057355,2651971,color,1C0 Shell,very light with cool pink undertones,https://www.ulta.com/p/double-wear-stay-in-pla...,1.0 oz,InStock,https://media.ultainc.com/i/ulta/2651971_sw?im...,https://media.ultainc.com/i/ulta/2651971?w=500...,7504fc4671712f74cee9be963f0736dc2a3054032beca7...,52.0,Estée Lauder,Double Wear Stay-in-Place Longwear Matte Found...,1.0,oz
2,pimprod2057355,2651972,color,1C1 Cool Bone,light with cool rosy-peach undertones,https://www.ulta.com/p/double-wear-stay-in-pla...,1.0 oz,InStock,https://media.ultainc.com/i/ulta/2651972_sw?im...,https://media.ultainc.com/i/ulta/2651972?w=500...,7504fc4671712f74cee9be963f0736dc2a3054032beca7...,52.0,Estée Lauder,Double Wear Stay-in-Place Longwear Matte Found...,1.0,oz
3,pimprod2057355,2651973,color,1N0 Porcelain,very light with neutral undertones,https://www.ulta.com/p/double-wear-stay-in-pla...,1.0 oz,InStock,https://media.ultainc.com/i/ulta/2651973_sw?im...,https://media.ultainc.com/i/ulta/2651973?w=500...,7504fc4671712f74cee9be963f0736dc2a3054032beca7...,52.0,Estée Lauder,Double Wear Stay-in-Place Longwear Matte Found...,1.0,oz
4,pimprod2057355,2651974,color,1N1 Ivory Nude,light with neutral peach undertones,https://www.ulta.com/p/double-wear-stay-in-pla...,1.0 oz,InStock,https://media.ultainc.com/i/ulta/2651974_sw?im...,https://media.ultainc.com/i/ulta/2651974?w=500...,7504fc4671712f74cee9be963f0736dc2a3054032beca7...,52.0,Estée Lauder,Double Wear Stay-in-Place Longwear Matte Found...,1.0,oz


In [ ]:
foundations.drop(columns=['availability', 'variant_type', 'size_text', 'price_per_ox', 'list_price', 'sale_price'], inplace=True)

In [21]:
foundations['price_per_oz'] = (foundations['price'] / foundations['size_value']).round(2)

In [28]:
foundations

,product_id,sku_id,variant_name,variant_description,variant_url,swatch_image_url,variant_image_url,ingredient_set_id,price,brand,product_name,size_value,size_unit,price_per_oz
0,pimprod2057355,2651970,0N1 Alabaster,lightest with neutral undertones,https://www.ulta.com/p/double-wear-stay-in-pla...,https://media.ultainc.com/i/ulta/2651970_sw?im...,https://media.ultainc.com/i/ulta/2651970?w=500...,7504fc4671712f74cee9be963f0736dc2a3054032beca7...,52.0,Estée Lauder,Double Wear Stay-in-Place Longwear Matte Found...,1.0,oz,52.0
1,pimprod2057355,2651971,1C0 Shell,very light with cool pink undertones,https://www.ulta.com/p/double-wear-stay-in-pla...,https://media.ultainc.com/i/ulta/2651971_sw?im...,https://media.ultainc.com/i/ulta/2651971?w=500...,7504fc4671712f74cee9be963f0736dc2a3054032beca7...,52.0,Estée Lauder,Double Wear Stay-in-Place Longwear Matte Found...,1.0,oz,52.0
2,pimprod2057355,2651972,1C1 Cool Bone,light with cool rosy-peach undertones,https://www.ulta.com/p/double-wear-stay-in-pla...,https://media.ultainc.com/i/ulta/2651972_sw?im...,https://media.ultainc.com/i/ulta/2651972?w=500...,7504fc4671712f74cee9be963f0736dc2a3054032beca7...,52.0,Estée Lauder,Double Wear Stay-in-Place Longwear Matte Found...,1.0,oz,52.0
3,pimprod2057355,2651973,1N0 Porcelain,very light with neutral undertones,https://www.ulta.com/p/double-wear-stay-in-pla...,https://media.ultainc.com/i/ulta/2651973_sw?im...,https://media.ultainc.com/i/ulta/2651973?w=500...,7504fc4671712f74cee9be963f0736dc2a3054032beca7...,52.0,Estée Lauder,Double Wear Stay-in-Place Longwear Matte Found...,1.0,oz,52.0
4,pimprod2057355,2651974,1N1 Ivory Nude,light with neutral peach undertones,https://www.ulta.com/p/double-wear-stay-in-pla...,https://media.ultainc.com/i/ulta/2651974_sw?im...,https://media.ultainc.com/i/ulta/2651974?w=500...,7504fc4671712f74cee9be963f0736dc2a3054032beca7...,52.0,Estée Lauder,Double Wear Stay-in-Place Longwear Matte Found...,1.0,oz,52.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
178,pimprod2046452,2627230,9W,deep with warm olive undertones,https://www.ulta.com/p/skin-tint-blurring-elix...,https://media.ultainc.com/i/ulta/2627230_sw?im...,https://media.ultainc.com/i/ulta/2627230?w=500...,9abca9c031fc1532a35d494e95e618818a25a76f0208de...,35.0,KYLIE COSMETICS,Skin Tint Blurring Elixir Foundation,1.0,oz,35.0
179,pimprod2046452,2627228,10C,"deep with cool, subtle rosy undertones",https://www.ulta.com/p/skin-tint-blurring-elix...,https://media.ultainc.com/i/ulta/2627228_sw?im...,https://media.ultainc.com/i/ulta/2627228?w=500...,9abca9c031fc1532a35d494e95e618818a25a76f0208de...,35.0,KYLIE COSMETICS,Skin Tint Blurring Elixir Foundation,1.0,oz,35.0
180,pimprod2046452,2627232,10N,deep dark with neutral undertones,https://www.ulta.com/p/skin-tint-blurring-elix...,https://media.ultainc.com/i/ulta/2627232_sw?im...,https://media.ultainc.com/i/ulta/2627232?w=500...,9abca9c031fc1532a35d494e95e618818a25a76f0208de...,35.0,KYLIE COSMETICS,Skin Tint Blurring Elixir Foundation,1.0,oz,35.0
181,pimprod2046452,2627231,10W,deep with warm undertones,https://www.ulta.com/p/skin-tint-blurring-elix...,https://media.ultainc.com/i/ulta/2627231_sw?im...,https://media.ultainc.com/i/ulta/2627231?w=500...,9abca9c031fc1532a35d494e95e618818a25a76f0208de...,35.0,KYLIE COSMETICS,Skin Tint Blurring Elixir Foundation,1.0,oz,35.0


In [ ]:
## Export 
foundations.to_csv(r"../data/processed_data/test/foundations/cleaned_foundations.csv", encoding='utf-8-sig')

In [ ]:
# python .\scripts\extract_swatch_colors.py --input "data\processed_data\test\foundations\cleaned_foundations.csv" --output "data\interim\foundation\swatch_foundations_sample.csv" 

## add the git

### Testing URLs

In [14]:
first_swatch_url = df_blush['swatch_image_url'].iloc[1]
second_swatch_url = df_blush['swatch_image_url'].iloc[2]

## Fetch and Crop Image
Ignoring the edges of the immage (which might contain white borders or shadows) and isolate the solic color in the middle.

In [29]:
def get_swatch_color(url: str, crop_fraction: float = 0.2):
  """Fetches an image & crops the center based on the crop_franction"""

  ## 1. Fetch the raw image bytes
  response = requests.get(url, headers={"User-Agent": "Mozilla/5.0"})
  response.raise_for_status()
  content = response.content

  ## 2. Open the image & ensure it has an Alpha (r,g,b,a)
  image = Image.open(BytesIO(content)).convert("RGBA")

  ## 3. Calculate the center crop box coordinates
  width, height = image.size # Get the image dimensions (80,80)
  crop_width = max(1, round(width * crop_fraction))
  crop_height = max(1, round(height * crop_fraction))
  # print(f"Image width: {width} & height: {height}")
  # print(f"Crop width: {crop_width}")
  # print(f"Crop height: {crop_height}")

  left = (width - crop_width) // 2
  top = (height - crop_height) // 2
  # print(f"Left: {left}")
  # print(f"Top: {top}")

  ## 4. Physically crop the image
  crop = image.crop((left, top, left + crop_width, top + crop_height))

  return crop

In [16]:
### --- TESTING get_swatch_color() ---

# Get the first swatch_image_url from the DataFrame

# print(f"Using the first swatch_image_url: {first_swatch_url}")

# Call get_swatch_color with the extracted URL
get_swatch_color(first_swatch_url)

In [17]:

dominated_color1 = get_swatch_color(first_swatch_url)
print(f"First URL: {display(dominated_color1)}") # to display the object dominated color



dominated_color2 = get_swatch_color(second_swatch_url)
print(f"Second URL: {display(dominated_color2)}") # to display the object dominated color

First URL: None


Second URL: None


## Pixel Measurement

In [30]:
def measure_pixels(crop_image: Image.Image):
  """Extracts median RGB values from a cropped PIL Image."""
  rgba_bytes = crop_image.tobytes()

  # Group bytes into standard RGBA pixels, ignoring highly transparent ones[cite: 11]
  opaque_pixels = [
      (rgba_bytes[i], rgba_bytes[i + 1], rgba_bytes[i + 2])
      for i in range(0, len(rgba_bytes), 4)
      if rgba_bytes[i + 3] > 15
  ]

  # Filter out pure white backgrounds (where all RGB values are >= 245)[cite: 11]
  non_white = [p for p in opaque_pixels if not all(c >= 245 for c in p)]

  # Fall back to opaque pixels if filtering out white removed everything[cite: 11]
  sampled = non_white if len(non_white) > 0 else opaque_pixels

  # Calculate the median for each color channel individually[cite: 11]
  red = int(median(pixel[0] for pixel in sampled))
  green = int(median(pixel[1] for pixel in sampled))
  blue = int(median(pixel[2] for pixel in sampled))

  color_measurement = ColorMeasurement(
      r=red,
      g=green,
      b=blue,
      hex_color=f"#{red:02x}{green:02x}{blue:02x}"
  )
  # display(color_measurement) # Automatically display the result
  return color_measurement

In [19]:
### --- TESTING measure_pixels() ---
# dominated_color1 = get_swatch_color(first_swatch_url)
measure1 = measure_pixels(dominated_color1)
# print(f"First URL: {display(dominated_color1)}") # to display the object dominated color
measure1

ColorMeasurement(r=232, g=136, b=145, hex_color='#e88891')

In [24]:
color10 = color_blush.copy()

In [31]:
## display the swatch_image_url using their extracted color codes (RGB)
def display_color_from_rgb(r, g, b, hex_code):
  """Generates an HTML display for a given RGB color."""
  color_box_html = f"<div style='width: 50px; height: 50px; background-color: rgb({r},{g},{b}); border: 1px solid black;'></div>"
  text_html = f"RGB: ({r}, {g}, {b})<br>Hex: {hex_code}"
  return HTML(f"<div style='display: flex; align-items: center;'>{color_box_html}<div style='margin-left: 10px;'>{text_html}</div></div>")

In [ ]:
# Iterate through the first 10 colors and display them
for index, row in color10.iterrows():
    r, g, b = int(row['rgb_r']), int(row['rgb_g']), int(row['rgb_b'])
    hex_color = row['hex_color']
    display(display_color_from_rgb(r, g, b, hex_color))


In [33]:
def get_color_family(lab_a, lab_b):
  """Classifies a color as 'Warm', 'Cool', or 'Neutral' based on lab_a and lab_b values."""
  # These thresholds are approximate and can be fine-tuned based on visual perception
  if lab_a > 15 and lab_b > 15:  # High red and yellow components
    return 'Warm'
  elif lab_a < -10 or lab_b < 0: # High green or blue components
    return 'Cool'
  else:
    return 'Neutral'

def get_color_value(lab_l):
  """Classifies a color as 'Low Value' (dark) or 'High Value' (light) based on lab_l value."""
  if lab_l < 50:
    return 'Low Value' # Darker colors
  else:
    return 'High Value' # Lighter colors



In [30]:
# Apply the classification functions to the color10 DataFrame
color10['color_family'] = color10.apply(lambda row: get_color_family(row['lab_a'], row['lab_b']), axis=1)
color10['color_value'] = color10['lab_l'].apply(get_color_value)

# Display the DataFrame with the new classification columns
display(color10[['rgb_r', 'rgb_g', 'rgb_b', 'hex_color', 'lab_l', 'lab_a', 'lab_b', 'color_family', 'color_value']])

,rgb_r,rgb_g,rgb_b,hex_color,lab_l,lab_a,lab_b,color_family,color_value
0,242,124,132,#f27c84,65.72,45.91,16.74,Warm,High Value
1,232,136,145,#e88891,67.14,37.59,11.20,Neutral,High Value
2,243,65,61,#f3413d,55.26,66.62,43.51,Warm,High Value
3,254,111,87,#fe6f57,64.27,52.96,40.25,Warm,High Value
4,252,74,132,#fc4a84,59.22,70.50,7.83,Neutral,High Value
...,...,...,...,...,...,...,...,...,...
1358,159,72,53,#9f4835,41.63,34.72,28.59,Warm,Low Value
1359,225,179,145,#e1b391,76.25,12.05,23.72,Neutral,High Value
1360,243,181,166,#f3b5a6,78.96,20.49,16.45,Warm,High Value
1361,247,187,159,#f7bb9f,80.64,17.98,22.59,Warm,High Value


In [31]:
# Get unique color families
unique_color_families = color10['color_family'].unique() # 3 families: cool, neutral, warm

for family in unique_color_families:
    display(HTML(f"<h3>Color Family: {family}</h3>"))
    family_colors = color10[color10['color_family'] == family]
    for index, row in family_colors.iterrows():
        r, g, b = int(row['rgb_r']), int(row['rgb_g']), int(row['rgb_b'])
        hex_color = row['hex_color']
        display(display_color_from_rgb(r, g, b, hex_color))

['Warm' 'Neutral' 'Cool']


In [32]:
family_colors = color10[color10['color_family'] == 'cool']
for index, row in family_colors.iterrows():
    r, g, b = int(row['rgb_r']), int(row['rgb_g']), int(row['rgb_b'])
    hex_color = row['hex_color']
    display(display_color_from_rgb(r, g, b, hex_color))

In [43]:
len(color10.loc[color10['color_family'] == 'warm'])

0

In [44]:
# Using the correct capitalization 'Warm'
num_warm_colors = len(color10.loc[color10['color_family'] == 'Warm'])
print(f"Number of 'Warm' colors: {num_warm_colors}")

# Display the 'Warm' colors if any exist
if num_warm_colors > 0:
    display(HTML("<h3>Warm Colors:</h3>"))
    warm_colors_df = color10[color10['color_family'] == 'Warm']
    for index, row in warm_colors_df.iterrows():
        r, g, b = int(row['rgb_r']), int(row['rgb_g']), int(row['rgb_b'])
        hex_color = row['hex_color']
        display(display_color_from_rgb(r, g, b, hex_color))
else:
    print("No 'Warm' colors found after correcting capitalization.")

Number of 'Warm' colors: 812
